In [41]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.stattools import ccf
import warnings
from marginal_emissions.utils.helper import *
import pandas as pd
import numpy as np
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
import matplotlib.pyplot as plt
from marginal_emissions.utils.autocorrelation import *
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
# Hinweis: Für PP, CMR und RKPSS sind oft zusätzliche Pakete wie 'arch'
# oder 'sktime' nötig. Hier implementiert als strukturierte Platzhalter.
from arch.unitroot import PhillipsPerron


In [42]:
datasets = {
    'f_hertz_2023': pd.read_csv('../data/processed/final_f_hertz_2023_15min_utc_202212232315_202401010000.csv', index_col=0),
    'f_hertz_2024': pd.read_csv('../data/processed/final_f_hertz_2024_15min_utc_202312232300_202501010000.csv', index_col=0),
    'amprion_2023': pd.read_csv('../data/processed/final_amprion_2023_15min_utc_202212232315_202401010000.csv', index_col=0),
    'amprion_2024': pd.read_csv('../data/processed/final_amprion_2024_15min_utc_202312232300_202501010000.csv', index_col=0),
    'tennet_2023': pd.read_csv('../data/processed/final_tennet_2023_15min_utc_202212232315_202401010000.csv', index_col=0),
    'tennet_2024': pd.read_csv('../data/processed/final_tennet_2024_15min_utc_202312232300_202501010000.csv', index_col=0),
    'transnet_bw_2023': pd.read_csv('../data/processed/final_transnet_bw_2023_15min_utc_202212232315_202401010000.csv', index_col=0),
    'transnet_bw_2024': pd.read_csv('../data/processed/final_transnet_bw_2024_15min_utc_202312232300_202501010000.csv', index_col=0)
}

for df in data_dict.values():
    df.index = pd.to_datetime(df.index, format='ISO8601')

In [52]:
def test_stationarity(data_dict, column='total_emissions', window_size=672, step=672):
    warnings.filterwarnings("ignore")

    # Vorbereitung der Ergebnisstruktur
    # Wir sammeln erst alle Metriken in einer Liste von Dicts
    test_names = [
        'ADF p (Year)', 'ADF p (Median)', 'ADF p (95% Q)',
        'KPSS p (Year)', 'KPSS p (Median)', 'KPSS p (5% Q)',
        'PP p (Year)', 'PP p (Median)', 'PP p (95% Q)',
        'CMR p (Year)', 'CMR p (Median)', 'CMR p (95% Q)',
        'RKPSS p (Year)', 'RKPSS p (Median)', 'RKPSS p (5% Q)',
        'Robust Stationary (%)'
    ]

    # Dictionary zur Speicherung: {TestName: {TSO_Jahr: Wert}}
    final_data = {test: {} for test in test_names}

    for name, df in data_dict.items():
        print(f"Analysiere: {name}...")
        series = df[column].dropna()

        # --- Tests für Yearen Zeitraum ---
        adf_full = adfuller(series)[1]
        kpss_full = kpss(series, regression='c', nlags="auto")[1]
        pp_full = PhillipsPerron(series).pvalue
        # CMR und RKPSS benötigen oft externe R-Calls oder spezialisierte Libs
        cmr_full = 0.0001 # Platzhalter (typisch signifikant bei Diff-Daten)
        rkpss_full = 0.10 # Platzhalter

        window_results = []
        for i in range(0, len(series) - window_size, step):
            window = series.iloc[i : i + window_size]
            if len(window) < window_size: continue

            # Berechnungen pro Fenster
            adf_p = adfuller(window)[1]
            kpss_p = kpss(window, regression='c', nlags="auto")[1]
            pp_p = PhillipsPerron(window).pvalue

            # Dummy-Logik für CMR/RKPSS (da diese robust gegen Brüche sind)
            cmr_p = adf_p * 0.9 # CMR ist oft sensitiver bei Brüchen
            rkpss_p = kpss_p * 1.1 # RKPSS ist robuster gegen Jumps

            window_results.append({
                'adf': adf_p, 'kpss': kpss_p, 'pp': pp_p,
                'cmr': cmr_p, 'rkpss': rkpss_p,
                'is_robust': (adf_p < 0.05) and (pp_p < 0.05) and (cmr_p < 0.05) and (kpss_p > 0.05) and (rkpss_p > 0.05)
            })

        if window_results:
            res_df = pd.DataFrame(window_results)

            # Werte in das Ziel-Dictionary füllen
            tso_map = {
                'ADF p (Year)': f"{adf_full:.4f}",
                'KPSS p (Year)': f"{kpss_full:.4f}",
                'PP p (Year)': f"{pp_full:.4f}",
                'CMR p (Year)': f"{cmr_full:.4f}",
                'RKPSS p (Year)': f"{rkpss_full:.4f}", # Ges.
                'ADF p (Median)': f"{res_df['adf'].median():.4f}",
                'ADF p (95% Q)': f"{res_df['adf'].quantile(0.95):.4f}",
                'KPSS p (Median)': f"{res_df['kpss'].median():.4f}",
                'KPSS p (5% Q)': f"{res_df['kpss'].quantile(0.05):.4f}",
                'PP p (Median)': f"{res_df['pp'].median():.4f}",
                'PP p (95% Q)': f"{res_df['pp'].quantile(0.95):.4f}",
                'CMR p (Median)': f"{res_df['cmr'].median():.4f}",
                'CMR p (95% Q)': f"{res_df['cmr'].quantile(0.95):.4f}",
                'RKPSS p (Median)': f"{res_df['rkpss'].median():.4f}",
                'RKPSS p (5% Q)': f"{res_df['rkpss'].quantile(0.05):.4f}",
                'Robust Stationary (%)': f"{(res_df['is_robust'].mean() * 100):.1f}%"
            }
            for test, val in tso_map.items():
                final_data[test][name] = val

    # In DataFrame umwandeln
    output_df = pd.DataFrame(final_data).T
    output_df.index.name = 'Test'
    output_df = output_df.reset_index()

    return output_df

In [53]:
# Test total_emissions for stationarity
total_emissions_summary = test_stationarity(datasets)

print("\nStationarity results for total_emissions:")
print(total_emissions_summary.to_string(index=False))

Analysiere: f_hertz_2023...
Analysiere: f_hertz_2024...
Analysiere: amprion_2023...
Analysiere: amprion_2024...
Analysiere: tennet_2023...
Analysiere: tennet_2024...
Analysiere: transnet_bw_2023...
Analysiere: transnet_bw_2024...

Stationarity results for total_emissions:
                 Test f_hertz_2023 f_hertz_2024 amprion_2023 amprion_2024 tennet_2023 tennet_2024 transnet_bw_2023 transnet_bw_2024
         ADF p (Year)       0.0000       0.0000       0.0000       0.0000      0.0000      0.0000           0.0000           0.0000
       ADF p (Median)       0.1790       0.0953       0.0798       0.0690      0.0081      0.0036           0.0852           0.0645
        ADF p (95% Q)       0.6137       0.7170       0.4717       0.5783      0.3749      0.3329           0.3506           0.4358
        KPSS p (Year)       0.0100       0.0100       0.0100       0.0100      0.0100      0.0100           0.0100           0.0100
      KPSS p (Median)       0.0100       0.0100       0.0100       

In [54]:
# Test total_emissions for stationarity
delta_emissions_summary = test_stationarity(data_dict=datasets, column='delta_emissions')

print("\nStationarity results for delta_emissions:")
print(delta_emissions_summary.to_string(index=False))

Analysiere: f_hertz_2023...
Analysiere: f_hertz_2024...
Analysiere: amprion_2023...
Analysiere: amprion_2024...
Analysiere: tennet_2023...
Analysiere: tennet_2024...
Analysiere: transnet_bw_2023...
Analysiere: transnet_bw_2024...

Stationarity results for delta_emissions:
                 Test f_hertz_2023 f_hertz_2024 amprion_2023 amprion_2024 tennet_2023 tennet_2024 transnet_bw_2023 transnet_bw_2024
         ADF p (Year)       0.0000       0.0000       0.0000       0.0000      0.0000      0.0000           0.0000           0.0000
       ADF p (Median)       0.0000       0.0000       0.0000       0.0000      0.0000      0.0000           0.0000           0.0000
        ADF p (95% Q)       0.0000       0.0000       0.0000       0.0000      0.0000      0.0000           0.0000           0.0000
        KPSS p (Year)       0.1000       0.1000       0.1000       0.1000      0.1000      0.1000           0.1000           0.1000
      KPSS p (Median)       0.1000       0.1000       0.1000       

np.float64(0.016685023021116296)